In [41]:
from google.colab import files
uploaded = files.upload()

Saving online_shopping.csv to online_shopping (1).csv


In [42]:
import numpy as np
import pandas as pd

df = pd.read_csv('online_shopping.csv')

In [43]:
y_list = []
for val in df['Revenue']:
    if val == True or val == 1 or val == 'True' or val == '1':
        y_list.append([1])
    else:
        y_list.append([0])
y = np.array(y_list)

In [44]:

categorical_cols = ['Month', 'VisitorType', 'Weekend']
numerical_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

encoded_rows = []
for idx, row in df.iterrows():
    row_encoded = []

    for col in numerical_cols:
        row_encoded.append(float(row[col]))

    for col in categorical_cols:
        value = row[col]
        unique_vals = df[col].unique()
        for u_val in unique_vals:
            if value == u_val:
                row_encoded.append(1.0)
            else:
                row_encoded.append(0.0)

    encoded_rows.append(row_encoded)

X = np.array(encoded_rows)
print(X)

[[ 0.  0.  0. ...  0.  1.  0.]
 [ 0.  0.  0. ...  0.  1.  0.]
 [ 0.  0.  0. ...  0.  1.  0.]
 ...
 [ 0.  0.  0. ...  0.  0.  1.]
 [ 4. 75.  0. ...  0.  1.  0.]
 [ 0.  0.  0. ...  0.  0.  1.]]


In [45]:

indices = np.random.permutation(len(X))

split_idx = int(0.8 * len(X))
train_idx = indices[:split_idx]
test_idx = indices[split_idx:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [46]:
print(f"Total Samples: {len(X)}")
print(f"Total Input Features (after manual encoding): {X.shape[1]}")
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Total Samples: 12330
Total Input Features (after manual encoding): 29
Train samples: 9864, Test samples: 2466


In [47]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Sigmoid Derivative
def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

In [48]:
def relu(z):
    return np.maximum(0, z)

# ReLU Derivative
def relu_derivative(z):
    return np.where(z > 0, 1.0, 0.0)

In [49]:
# Tanh Activation & Derivative
def tanh(z):
    return (np.exp(z) - np.exp(-z)) / (np.exp(z) + np.exp(-z))

def tanh_derivative(z):
    t = tanh(z)
    return 1.0 - t**2

In [50]:
# LeakyReLU Activation & Derivative (Manual Formula)
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

In [51]:
# Binary Cross Entropy Loss
def compute_loss(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Cost Function
def compute_cost(y_true, y_pred):
    m = len(y_true)
    total_loss = np.sum(compute_loss(y_true, y_pred))
    cost = total_loss / m
    return cost


In [52]:
def initialize_parameters(input_size, hidden1_size, hidden2_size, output_size):
    np.random.seed(42)

    # Hidden Layer 1 Weights & Biases
    W1 = np.random.randn(input_size, hidden1_size) * 0.01
    b1 = np.zeros((1, hidden1_size))

    # Hidden Layer 2 Weights & Biases
    W2 = np.random.randn(hidden1_size, hidden2_size) * 0.01
    b2 = np.zeros((1, hidden2_size))

    # Output Layer Weights & Biases
    W3 = np.random.randn(hidden2_size, output_size) * 0.01
    b3 = np.zeros((1, output_size))

    return W1, b1, W2, b2, W3, b3

In [53]:
# Forward Propagation
def forward_propagation(X, W1, b1, W2, b2, W3, b3):
    # Hidden Layer 1
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    # Hidden Layer 2
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)

    # Output Layer
    Z3 = np.dot(A2, W3) + b3
    A3 = sigmoid(Z3)

    return Z1, A1, Z2, A2, Z3, A3

In [54]:
# ৩. Backward Propagation & Weight Update
def backward_propagation(X, y, Z1, A1, Z2, A2, Z3, A3, W1, b1, W2, b2, W3, b3, learning_rate):
    m = X.shape[0]

    # Output Layer Gradient
    dZ3 = A3 - y
    dW3 = np.dot(A2.T, dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # Hidden Layer 2 Gradient
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Hidden Layer 1 Gradient
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    # Update Weights & Biases
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

    return W1, b1, W2, b2, W3, b3

print("Multi-Layer Functions Ready!")

Multi-Layer Functions Ready!


In [55]:
input_dim = X_train.shape[1]
hidden1_dim = 16
hidden2_dim = 8
output_dim = 1
learning_rate = 0.01
epochs = 50


In [56]:
def predict_and_evaluate(X, y, W1, b1, W2, b2, W3, b3):
    _, _, _, _, _, predictions = forward_propagation(X, W1, b1, W2, b2, W3, b3)
    pred_labels = (predictions >= 0.5).astype(int)
    accuracy = np.mean(pred_labels == y) * 100
    return accuracy

In [57]:
# --- 1. Batch Gradient Descent ---

W1, b1, W2, b2, W3, b3 = initialize_parameters(input_dim, hidden1_dim, hidden2_dim, output_dim)

print("--- Running Batch Gradient Descent ---")
for epoch in range(epochs):
    Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X_train, W1, b1, W2, b2, W3, b3)
    cost = compute_cost(y_train, A3)

    W1, b1, W2, b2, W3, b3 = backward_propagation(
        X_train, y_train, Z1, A1, Z2, A2, Z3, A3,
        W1, b1, W2, b2, W3, b3, learning_rate
    )
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Cost: {cost:.4f}")

# Batch GD Evaluation
test_acc = predict_and_evaluate(X_test, y_test, W1, b1, W2, b2, W3, b3)
print(f"Batch GD Test Accuracy: {test_acc:.2f}%\n")

--- Running Batch Gradient Descent ---
Epoch 1/50 - Cost: 0.6930
Epoch 10/50 - Cost: 0.6818
Epoch 20/50 - Cost: 0.6645
Epoch 30/50 - Cost: 0.6222
Epoch 40/50 - Cost: 0.6131
Epoch 50/50 - Cost: 0.6079
Batch GD Test Accuracy: 84.35%



In [58]:
# --- 2. Mini-Batch Gradient Descent ---
batch_size = 32

W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb = initialize_parameters(input_dim, hidden1_dim, hidden2_dim, output_dim)
m = X_train.shape[0]

print("--- Running Mini-Batch Gradient Descent ---")
for epoch in range(epochs):
    permutation = np.random.permutation(m)
    X_shuffled = X_train[permutation]
    y_shuffled = y_train[permutation]

    num_complete_batches = m // batch_size
    for k in range(num_complete_batches):
        X_batch = X_shuffled[k * batch_size : (k + 1) * batch_size]
        y_batch = y_shuffled[k * batch_size : (k + 1) * batch_size]

        Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X_batch, W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb)
        W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb = backward_propagation(
            X_batch, y_batch, Z1, A1, Z2, A2, Z3, A3,
            W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb, learning_rate
        )

    if m % batch_size != 0:
        X_batch = X_shuffled[num_complete_batches * batch_size :]
        y_batch = y_shuffled[num_complete_batches * batch_size :]
        Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X_batch, W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb)
        W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb = backward_propagation(
            X_batch, y_batch, Z1, A1, Z2, A2, Z3, A3,
            W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb, learning_rate
        )

    _, _, _, _, _, A3_full = forward_propagation(X_train, W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb)
    cost = compute_cost(y_train, A3_full)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Cost: {cost:.4f}")

# Mini-Batch GD Evaluation
test_acc_mb = predict_and_evaluate(X_test, y_test, W1_mb, b1_mb, W2_mb, b2_mb, W3_mb, b3_mb)
print(f"Mini-Batch GD Test Accuracy: {test_acc_mb:.2f}%\n")

--- Running Mini-Batch Gradient Descent ---
Epoch 1/50 - Cost: 0.5130
Epoch 10/50 - Cost: 0.4300
Epoch 20/50 - Cost: 0.4299
Epoch 30/50 - Cost: 0.4286
Epoch 40/50 - Cost: 0.4205
Epoch 50/50 - Cost: 0.4230
Mini-Batch GD Test Accuracy: 84.35%



In [59]:
# --- 3. Stochastic Gradient Descent (SGD) ---

W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd = initialize_parameters(input_dim, hidden1_dim, hidden2_dim, output_dim)
m = X_train.shape[0]

print("--- Running Stochastic Gradient Descent (SGD) ---")
for epoch in range(epochs):
    permutation = np.random.permutation(m)
    X_shuffled = X_train[permutation]
    y_shuffled = y_train[permutation]

    for i in range(m):
        X_sample = X_shuffled[i:i+1]
        y_sample = y_shuffled[i:i+1]

        Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X_sample, W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd)
        W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd = backward_propagation(
            X_sample, y_sample, Z1, A1, Z2, A2, Z3, A3,
            W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd, learning_rate
        )

    _, _, _, _, _, A3_full = forward_propagation(X_train, W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd)
    cost = compute_cost(y_train, A3_full)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Cost: {cost:.4f}")

# SGD Evaluation
test_acc_sgd = predict_and_evaluate(X_test, y_test, W1_sgd, b1_sgd, W2_sgd, b2_sgd, W3_sgd, b3_sgd)
print(f"SGD Test Accuracy: {test_acc_sgd:.2f}%\n")

--- Running Stochastic Gradient Descent (SGD) ---
Epoch 1/50 - Cost: 0.4320
Epoch 10/50 - Cost: 0.4208
Epoch 20/50 - Cost: 0.4212
Epoch 30/50 - Cost: 0.4206
Epoch 40/50 - Cost: 0.4210
Epoch 50/50 - Cost: 0.4205
SGD Test Accuracy: 84.35%



In [61]:
import numpy as np
import pandas as pd

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return np.where(z > 0, 1.0, 0.0)

def tanh(z):
    z = np.clip(z, -500, 500)
    return (np.exp(z) - np.exp(-z)) / (np.exp(z) + np.exp(-z))

def tanh_derivative(z):
    t = tanh(z)
    return 1.0 - t**2

def compute_cost(y_true, y_pred):
    m = len(y_true)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    total_loss = np.sum(-(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)))
    return total_loss / m

def initialize_parameters(input_size, hidden1_size, hidden2_size, output_size):
    np.random.seed(42)
    W1 = np.random.randn(input_size, hidden1_size) * 0.05
    b1 = np.zeros((1, hidden1_size))
    W2 = np.random.randn(hidden1_size, hidden2_size) * 0.05
    b2 = np.zeros((1, hidden2_size))
    W3 = np.random.randn(hidden2_size, output_size) * 0.05
    b3 = np.zeros((1, output_size))
    return W1, b1, W2, b2, W3, b3

input_dim = X_train.shape[1]
hidden1_dim = 24
hidden2_dim = 12
output_dim = 1
learning_rate = 0.05
epochs = 40

optimizers = ['Batch', 'Mini-Batch', 'SGD']
activations = ['relu', 'tanh', 'sigmoid']
results = []

print("--- Starting 9 Combinations Training & Evaluation ---")

for opt in optimizers:
    for act in activations:
        W1, b1, W2, b2, W3, b3 = initialize_parameters(input_dim, hidden1_dim, hidden2_dim, output_dim)
        final_cost = 0.0

        def activate(z, func_name):
            if func_name == 'relu': return relu(z)
            elif func_name == 'tanh': return tanh(z)
            else: return sigmoid(z)

        def activate_derivative(z, func_name):
            if func_name == 'relu': return relu_derivative(z)
            elif func_name == 'tanh': return tanh_derivative(z)
            else: return sigmoid_derivative(z)

        # --- Batch Gradient Descent ---
        if opt == 'Batch':
            for epoch in range(epochs):
                Z1 = np.dot(X_train, W1) + b1
                A1 = activate(Z1, act)
                Z2 = np.dot(A1, W2) + b2
                A2 = activate(Z2, act)
                Z3 = np.dot(A2, W3) + b3
                A3 = sigmoid(Z3)

                final_cost = compute_cost(y_train, A3)
                m = X_train.shape[0]

                dZ3 = A3 - y_train
                dW3 = np.dot(A2.T, dZ3) / m
                db3 = np.sum(dZ3, axis=0, keepdims=True) / m

                dA2 = np.dot(dZ3, W3.T)
                dZ2 = dA2 * activate_derivative(Z2, act)
                dW2 = np.dot(A1.T, dZ2) / m
                db2 = np.sum(dZ2, axis=0, keepdims=True) / m

                dA1 = np.dot(dZ2, W2.T)
                dZ1 = dA1 * activate_derivative(Z1, act)
                dW1 = np.dot(X_train.T, dZ1) / m
                db1 = np.sum(dZ1, axis=0, keepdims=True) / m

                W1 -= learning_rate * dW1
                b1 -= learning_rate * db1
                W2 -= learning_rate * dW2
                b2 -= learning_rate * db2
                W3 -= learning_rate * dW3
                b3 -= learning_rate * db3

        # --- Mini-Batch Gradient Descent ---
        elif opt == 'Mini-Batch':
            batch_size = 64
            m = X_train.shape[0]
            for epoch in range(epochs):
                permutation = np.random.permutation(m)
                X_shuffled = X_train[permutation]
                y_shuffled = y_train[permutation]
                num_complete_batches = m // batch_size

                for k in range(num_complete_batches):
                    X_batch = X_shuffled[k * batch_size : (k + 1) * batch_size]
                    y_batch = y_shuffled[k * batch_size : (k + 1) * batch_size]

                    Z1 = np.dot(X_batch, W1) + b1
                    A1 = activate(Z1, act)
                    Z2 = np.dot(A1, W2) + b2
                    A2 = activate(Z2, act)
                    Z3 = np.dot(A2, W3) + b3
                    A3 = sigmoid(Z3)

                    mb_m = X_batch.shape[0]
                    dZ3 = A3 - y_batch
                    dW3 = np.dot(A2.T, dZ3) / mb_m
                    db3 = np.sum(dZ3, axis=0, keepdims=True) / mb_m

                    dA2 = np.dot(dZ3, W3.T)
                    dZ2 = dA2 * activate_derivative(Z2, act)
                    dW2 = np.dot(A1.T, dZ2) / mb_m
                    db2 = np.sum(dZ2, axis=0, keepdims=True) / mb_m

                    dA1 = np.dot(dZ2, W2.T)
                    dZ1 = dA1 * activate_derivative(Z1, act)
                    dW1 = np.dot(X_batch.T, dZ1) / mb_m
                    db1 = np.sum(dZ1, axis=0, keepdims=True) / mb_m

                    W1 -= learning_rate * dW1
                    b1 -= learning_rate * db1
                    W2 -= learning_rate * dW2
                    b2 -= learning_rate * db2
                    W3 -= learning_rate * dW3
                    b3 -= learning_rate * db3

            Z1 = np.dot(X_train, W1) + b1
            A1 = activate(Z1, act)
            Z2 = np.dot(A1, W2) + b2
            A2 = activate(Z2, act)
            Z3 = np.dot(A2, W3) + b3
            A3 = sigmoid(Z3)
            final_cost = compute_cost(y_train, A3)

        # --- Stochastic Gradient Descent (SGD) ---
        elif opt == 'SGD':
            m = X_train.shape[0]
            for epoch in range(epochs):
                permutation = np.random.permutation(m)
                X_shuffled = X_train[permutation]
                y_shuffled = y_train[permutation]

                for i in range(min(200, m)):
                    X_sample = X_shuffled[i:i+1]
                    y_sample = y_shuffled[i:i+1]

                    Z1 = np.dot(X_sample, W1) + b1
                    A1 = activate(Z1, act)
                    Z2 = np.dot(A1, W2) + b2
                    A2 = activate(Z2, act)
                    Z3 = np.dot(A2, W3) + b3
                    A3 = sigmoid(Z3)

                    dZ3 = A3 - y_sample
                    dW3 = np.dot(A2.T, dZ3)
                    db3 = dZ3

                    dA2 = np.dot(dZ3, W3.T)
                    dZ2 = dA2 * activate_derivative(Z2, act)
                    dW2 = np.dot(A1.T, dZ2)
                    db2 = dZ2

                    dA1 = np.dot(dZ2, W2.T)
                    dZ1 = dA1 * activate_derivative(Z1, act)
                    dW1 = np.dot(X_sample.T, dZ1)
                    db1 = dZ1

                    W1 -= learning_rate * dW1
                    b1 -= learning_rate * db1
                    W2 -= learning_rate * dW2
                    b2 -= learning_rate * db2
                    W3 -= learning_rate * dW3
                    b3 -= learning_rate * db3

            Z1 = np.dot(X_train, W1) + b1
            A1 = activate(Z1, act)
            Z2 = np.dot(A1, W2) + b2
            A2 = activate(Z2, act)
            Z3 = np.dot(A2, W3) + b3
            A3 = sigmoid(Z3)
            final_cost = compute_cost(y_train, A3)

        # Test Evaluation
        Z1_test = np.dot(X_test, W1) + b1
        A1_test = activate(Z1_test, act)
        Z2_test = np.dot(A1_test, W2) + b2
        A2_test = activate(Z2_test, act)
        Z3_test = np.dot(A2_test, W3) + b3
        predictions = sigmoid(Z3_test)
        pred_labels = (predictions >= 0.5).astype(int)
        accuracy = np.mean(pred_labels == y_test) * 100

        results.append({
            "Optimizer": opt,
            "Activation": act.upper(),
            "Final Cost": round(float(final_cost), 4),
            "Test Accuracy (%)": round(float(accuracy), 2)
        })
results_df = pd.DataFrame(results)

def highlight_best(s):
    is_max = s == s.max()
    return ['font-weight: bold; background-color: #d4edda' if v else '' for v in is_max]

print("\n==============================================")
print("   FINAL COMPARISON TABLE (9 COMBINATIONS)    ")
print("==============================================")

styled_df = results_df.style.apply(highlight_best, subset=['Test Accuracy (%)'])
display(styled_df)

--- Starting 9 Combinations Training & Evaluation ---

   FINAL COMPARISON TABLE (9 COMBINATIONS)    


,Optimizer,Activation,Final Cost,Test Accuracy (%)
0,Batch,RELU,0.533600,84.390000
1,Batch,TANH,0.457900,84.350000
2,Batch,SIGMOID,0.440400,84.350000
3,Mini-Batch,RELU,0.420700,84.350000
4,Mini-Batch,TANH,0.416600,84.390000
5,Mini-Batch,SIGMOID,0.415000,84.350000
6,SGD,RELU,0.425500,84.350000
7,SGD,TANH,0.437200,84.350000
8,SGD,SIGMOID,0.438000,84.350000
